# 🎯 Bài 4: Phát hiện vật thể - Haar Cascades & YOLOv8
**Môn học**: E1402 - Digital and Computer Vision | **Phiên bản**: Google Colab ☁️

**Thời gian**: ~60 phút | **Trình độ**: Người mới bắt đầu

---
## 🎯 Mục tiêu bài học
1. Hiểu nguyên lý phát hiện vật thể (Object Detection)
2. Phát hiện khuôn mặt & mắt bằng Haar Cascade
3. Phát hiện 80+ loại vật thể bằng YOLOv8
4. So sánh Haar vs YOLO
5. Upload ảnh & test webcam trực tiếp

> 💡 Nhấn ▶️ Play từng ô code

---
## 📖 Object Detection là gì?

**Object Detection** = Tìm VÀ nhận dạng vật thể trong ảnh.

```
Classification:    Detection:              Segmentation:
┌──────────┐      ┌──────────┐            ┌──────────┐
│          │      │ ┌──┐     │            │ ████     │
│   🐱     │      │ │🐱│ ┌─┐ │            │ ████ ██  │
│          │      │ └──┘ │🐕│ │            │      ██  │
│   "mèo"  │      │     └─┘ │            │          │
└──────────┘      └──────────┘            └──────────┘
  1 nhãn           Khung bao + nhãn        Mask pixel
```

| | Haar Cascade (2001) | YOLOv8 (2023) |
|---|---|---|
| **Phương pháp** | Machine Learning cổ điển | Deep Learning (CNN) |
| **Tốc độ** | ⚡ Rất nhanh (CPU) | ⚡ Nhanh (GPU tốt hơn) |
| **Số loại** | 1 loại/mô hình | 80+ loại cùng lúc |
| **Độ chính xác** | Tốt cho mặt | Xuất sắc |

## ⚙️ Cài đặt

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen, Request
from google.colab import files
from collections import Counter

def tai_anh(url):
    """Tải ảnh từ URL"""
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    resp = urlopen(req)
    arr = np.asarray(bytearray(resp.read()), dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def hien_thi(ds_anh, ds_ten, figsize=(16, 5)):
    """Hiển thị nhiều ảnh cạnh nhau"""
    fig, axes = plt.subplots(1, len(ds_anh), figsize=figsize)
    if len(ds_anh) == 1: axes = [axes]
    for ax, anh, ten in zip(axes, ds_anh, ds_ten):
        if len(anh.shape) == 2: ax.imshow(anh, cmap='gray')
        else: ax.imshow(cv2.cvtColor(anh, cv2.COLOR_BGR2RGB))
        ax.set_title(ten, fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Sẵn sàng!')

---
# 👤 Phần 1: Haar Cascade - Phát hiện khuôn mặt

### Haar Cascade hoạt động thế nào?

Viola & Jones (2001) - thuật toán object detection real-time đầu tiên:

1. **Đặc trưng Haar**: Mẫu sáng/tối đơn giản (VD: mắt tối hơn má)
2. **Cascade**: Nhiều giai đoạn kiểm tra → loại nhanh vùng không phải mặt

```
Đặc trưng Haar:  ┌████┬░░░░┐   VD: Vùng mắt TỐI hơn vùng trán
                  │████│░░░░│   → Haar phát hiện sự chênh lệch này
                  └────┴────┘
```

**Cascade có sẵn**: mặt, mắt, nụ cười, mặt mèo, toàn thân...

In [ ]:
# Tải cascade khuôn mặt + mắt (có sẵn trong OpenCV)
cascade_mat = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
cascade_mắt = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_eye.xml')

# Tải ảnh nhóm người
anh_nhom = tai_anh('https://images.unsplash.com/photo-1529156069898-49953e39b3ac?w=640')
if anh_nhom is None:
    anh_nhom = np.ones((400, 600, 3), dtype=np.uint8) * 200
    for x, y in [(150,150),(300,160),(450,140)]:
        cv2.ellipse(anh_nhom, (x,y), (40,50), 0, 0, 360, (180,150,130), -1)

xam = cv2.cvtColor(anh_nhom, cv2.COLOR_BGR2GRAY)

# ===== THAM SỐ =====
so_lan_can = 5   # Thử: 3 (nhiều), 8 (chính xác)
# ===================

ds_mat = cascade_mat.detectMultiScale(xam, 1.1, so_lan_can, minSize=(30,30))

# Vẽ kết quả: mặt (xanh lá) + mắt (xanh dương)
kq = anh_nhom.copy()
for (x, y, w, h) in ds_mat:
    cv2.rectangle(kq, (x,y), (x+w,y+h), (0,255,0), 3)
    roi_x = xam[y:y+h, x:x+w]
    roi_m = kq[y:y+h, x:x+w]
    mat = cascade_mắt.detectMultiScale(roi_x, 1.1, 5)
    for (ex,ey,ew,eh) in mat:
        cv2.rectangle(roi_m, (ex,ey), (ex+ew,ey+eh), (255,0,0), 2)

hien_thi([anh_nhom, kq],
         ['Ảnh gốc', f'Haar: {len(ds_mat)} mặt (xanh lá) + mắt (xanh dương)'],
         figsize=(16, 7))
print(f'👤 Phát hiện {len(ds_mat)} khuôn mặt!')
print('✏️ THỬ: Đổi so_lan_can = 3 (nhiều) hoặc 8 (ít, chính xác hơn)')

---
# 🚀 Phần 2: YOLOv8 - Object Detection hiện đại

**YOLO** = You Only Look Once

```
Haar: Trượt cửa sổ qua từng vị trí → CHẬM
YOLO: Nhìn TOÀN BỘ ảnh 1 lần → NHANH
```

| Mô hình | Dung lượng | Tốc độ | Phù hợp |
|---|---|---|---|
| `yolov8n` ⭐ | 6 MB | ⚡⚡⚡ | Di động, real-time |
| `yolov8s` | 22 MB | ⚡⚡ | Edge devices |
| `yolov8m` | 52 MB | ⚡ | Sử dụng chung |

Kết quả: **Bounding Box** + **Tên vật thể** + **Độ tin cậy** (0-100%)

Chúng ta dùng `yolov8n` (Nano) - nhỏ nhất, nhanh nhất!

In [ ]:
from ultralytics import YOLO
from collections import Counter

model = YOLO('yolov8n.pt')  # Tải mô hình Nano (~6MB)
print(f'✅ YOLOv8 Nano - phát hiện {len(model.names)} loại vật thể')
for i in range(0, len(model.names), 5):
    d = '  '
    for j in range(i, min(i+5, len(model.names))):
        d += f'{j:2d}.{model.names[j]:15s}'
    print(d)

In [ ]:
# Test YOLO trên ảnh mẫu
anh_test = tai_anh('https://ultralytics.com/images/bus.jpg')
if anh_test is None:
    anh_test = np.ones((480, 640, 3), dtype=np.uint8) * 180

kq = model(anh_test, verbose=False)
ve = kq[0].plot()
hien_thi([anh_test, ve],
         ['Ảnh gốc', f'YOLO: {len(kq[0].boxes)} vật thể'], figsize=(16, 7))

print('\n📋 Chi tiết:')
for box in kq[0].boxes:
    ten = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f'  {ten:>12s}: {conf:.1%}')

In [ ]:
# Test nhiều ảnh
ds = [('https://ultralytics.com/images/zidane.jpg', 'Thể thao'),
      ('https://ultralytics.com/images/bus.jpg', 'Giao thông')]
fig, axes = plt.subplots(1, len(ds), figsize=(18, 7))
for ax, (url, ten) in zip(axes, ds):
    anh = tai_anh(url)
    if anh is not None:
        r = model(anh, verbose=False)
        ax.imshow(cv2.cvtColor(r[0].plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f'{ten} ({len(r[0].boxes)} vật thể)', fontsize=13, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## ⚔️ So sánh Haar vs YOLO

In [ ]:
# So sánh Haar vs YOLO trên cùng ảnh
anh_ss = tai_anh('https://ultralytics.com/images/zidane.jpg')
if anh_ss is not None:
    haar_kq = anh_ss.copy()
    xam_ss = cv2.cvtColor(anh_ss, cv2.COLOR_BGR2GRAY)
    mat_ss = cascade_mat.detectMultiScale(xam_ss, 1.1, 5, minSize=(30,30))
    for (x,y,w,h) in mat_ss:
        cv2.rectangle(haar_kq, (x,y), (x+w,y+h), (0,255,0), 3)
    yolo_r = model(anh_ss, verbose=False)
    hien_thi([haar_kq, yolo_r[0].plot()],
             [f'Haar: {len(mat_ss)} (chỉ mặt)',
              f'YOLO: {len(yolo_r[0].boxes)} (mọi loại)'], figsize=(18, 7))
    print(f'📊 Haar: {len(mat_ss)} | YOLO: {len(yolo_r[0].boxes)}')

---
## 📤 Upload ảnh của bạn!

In [ ]:
print('📤 Nhấn Choose Files...')
uploaded = files.upload()
for ten, dl in uploaded.items():
    arr = np.frombuffer(dl, dtype=np.uint8)
    anh_sv = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    break
kq = model(anh_sv, verbose=False)
hien_thi([anh_sv, kq[0].plot()],
         ['Ảnh của bạn', f'YOLO: {len(kq[0].boxes)} vật thể'], figsize=(16,7))
dem = Counter(model.names[int(b.cls[0])] for b in kq[0].boxes)
for t, s in dem.most_common(): print(f'  {t}: {s}')

---
## 📹 Webcam YOLO (Colab)

Chụp ảnh trực tiếp từ webcam trình duyệt → YOLO phát hiện!

> ⚠️ Nhấn **Allow** khi trình duyệt hỏi quyền camera

### Chụp 1 ảnh

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
import base64, time

js = Javascript('''
async function chup() {
    const v = document.createElement("video");
    const s = await navigator.mediaDevices.getUserMedia({video:true});
    v.srcObject = s; await v.play();
    await new Promise(r => setTimeout(r, 1000));
    const c = document.createElement("canvas");
    c.width=v.videoWidth; c.height=v.videoHeight;
    c.getContext("2d").drawImage(v,0,0);
    s.getTracks().forEach(t=>t.stop());
    return c.toDataURL("image/jpeg",0.8);
}
''')
display(js)
print('📸 Đang mở webcam...')
data = eval_js('chup()')
binary = base64.b64decode(data.split(',')[1])
frame = cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)
kq = model(frame, verbose=False)
hien_thi([frame, kq[0].plot()],
         ['📸 Webcam', f'🎯 YOLO: {len(kq[0].boxes)} vật thể'], figsize=(16,7))
dem = Counter(model.names[int(b.cls[0])] for b in kq[0].boxes)
for t,s in dem.most_common(): print(f'  {t}: {s}')

### Chụp liên tục (multi-shot)

In [ ]:
so_anh = 5  # Số frame

js2 = Javascript('''
window._s=null;window._v=null;
async function mo(){if(window._s)return;const v=document.createElement("video");
v.style.display="none";document.body.appendChild(v);
const s=await navigator.mediaDevices.getUserMedia({video:{width:640,height:480}});
v.srcObject=s;await v.play();window._s=s;window._v=v;}
async function snap(){const c=document.createElement("canvas");
c.width=window._v.videoWidth;c.height=window._v.videoHeight;
c.getContext("2d").drawImage(window._v,0,0);return c.toDataURL("image/jpeg",0.7);}
async function tat(){if(window._s){window._s.getTracks().forEach(t=>t.stop());
window._v.remove();window._s=null;}}
''')
display(js2)
eval_js('mo()'); time.sleep(1)
print(f'📹 Chụp {so_anh} frames...')
frames = []
for i in range(so_anh):
    d = eval_js('snap()')
    f = cv2.imdecode(np.frombuffer(base64.b64decode(d.split(',')[1]),np.uint8),cv2.IMREAD_COLOR)
    t0=time.time(); r=model(f,verbose=False); dt=(time.time()-t0)*1000
    frames.append((r[0].plot(),len(r[0].boxes),dt))
    print(f'  Frame {i+1}: {len(r[0].boxes)} vật thể ({dt:.0f}ms)')
    time.sleep(0.3)
eval_js('tat()')
cols=min(3,len(frames)); rows=(len(frames)+cols-1)//cols
fig,axes=plt.subplots(rows,cols,figsize=(6*cols,5*rows))
ax=axes.flatten() if hasattr(axes,'flatten') else [axes]
for i,(a,n,m) in enumerate(frames):
    ax[i].imshow(cv2.cvtColor(a,cv2.COLOR_BGR2RGB))
    ax[i].set_title(f'Frame {i+1}: {n} ({m:.0f}ms)',fontsize=12,fontweight='bold')
    ax[i].axis('off')
for j in range(len(frames),len(ax)): ax[j].axis('off')
plt.suptitle('📹 YOLO Webcam',fontsize=16,fontweight='bold')
plt.tight_layout(); plt.show()

---
## 📊 Tổng kết Bài 4

| | Haar Cascade | YOLOv8 |
|---|---|---|
| Nguyên lý | Đặc trưng Haar + Cascade | CNN Deep Learning |
| Số loại | 1/mô hình | 80+ cùng lúc |
| Tốc độ | Cực nhanh (CPU) | Nhanh (GPU tốt hơn) |
| Chính xác | Tốt cho mặt | Xuất sắc |

### 🎉 Hoàn thành 4 bài Computer Vision!
1. ✅ Ảnh kỹ thuật số, pixel, RGB, HSV
2. ✅ Bộ lọc, histogram, ngưỡng hóa, Canny
3. ✅ K-Means, ORB, phép hình thái
4. ✅ Haar Cascade & YOLOv8 🚀